# 06 — Redis Cache
Explore @cached_node, TTLs, in-memory fallback, and cache invalidation.

In [ ]:
import sys; sys.path.insert(0, '/home/claude/codebase/code/src')
import os; os.environ['ENABLE_MOCK']='true'; os.environ['REDIS_ENABLED']='false'

## Cache primitives

In [ ]:
from core.cache import get_client, make_key, cache_get, cache_set, _fallback
import json

# In test mode Redis is disabled — falls back to in-memory dict
client = get_client()
print('Redis client:', client)   # None in test mode
print('Fallback cache type:', type(_fallback))

## Cache key generation

In [ ]:
key1 = make_key('information_agent',
    query='What is GRR?',
    data_products=['retention'],
    time_range='last_30_days'
)
key2 = make_key('information_agent',
    query='What is GRR?',
    data_products=['retention'],
    time_range='last_30_days'
)
key3 = make_key('information_agent',
    query='What is CAC?',  # different query
    data_products=['cac'],
    time_range='last_30_days'
)
print('Key 1:', key1)
print('Key 2:', key2)
print('Keys match for same input:', key1 == key2)
print('Key 3:', key3, '(different query)')

## Cache set/get round-trip

In [ ]:
_fallback.clear()
cache_set(None, key1, {'result': 'GRR is 92.5%'}, ttl=60)
cached = cache_get(None, key1)
print('Stored:', {'result': 'GRR is 92.5%'})
print('Retrieved:', cached)
print('Match:', cached == {'result': 'GRR is 92.5%'})

## @cached_node decorator in action

In [ ]:
import time
from graph.state import initial_state
from graph.graph import copilot_graph
import core.cache as cache_mod

cache_mod._fallback.clear()
cache_mod._client = None

state = initial_state(query='What is GRR?', data_products=['retention'], thread_id='cache-test')
cfg = {'configurable': {'thread_id': 'cache-test'}}

# First call — cache miss
t0 = time.monotonic()
r1 = copilot_graph.invoke(state, config=cfg)
t1 = time.monotonic()
print(f'First call (cache miss):  {(t1-t0)*1000:.1f}ms')

# Second call — cache hit (should be faster)
t2 = time.monotonic()
r2 = copilot_graph.invoke(state, config=cfg)
t3 = time.monotonic()
print(f'Second call (cache hit):  {(t3-t2)*1000:.1f}ms')
print(f'Cache entries: {len(cache_mod._fallback)}')

## Node TTLs

In [ ]:
print('Node TTL configuration:')
print('  information_agent: 1800s (30 min) — SQL query results')
print('  knowledge_agent:   7200s  (2 hrs) — RAG document retrieval')
print('  metadata_agent:    3600s  (1 hr)  — Collibra metadata')
print('  capacity_agent:    NOT CACHED — Jira tickets change frequently')
print('  synthesizer_node:  NOT CACHED — personalised LLM output')

## Cache invalidation

In [ ]:
from core.cache import invalidate_pattern

cache_mod._fallback.clear()
cache_set(None, 'information_agent:abc123', {'data': 1}, ttl=300)
cache_set(None, 'information_agent:def456', {'data': 2}, ttl=300)
cache_set(None, 'knowledge_agent:xyz789', {'data': 3}, ttl=300)

print('Before invalidation:', list(cache_mod._fallback.keys()))
deleted = invalidate_pattern(None, 'information_agent:*')
print(f'Deleted {deleted} information_agent entries')
print('After invalidation:', list(cache_mod._fallback.keys()))